# 🕵️ CaseBoard — a GraphRAG Detective
**Just click: Runtime ▸ Run all.** Everything is free (no API keys). Takes ~3–6 minutes the first time.

What you will get at the bottom: a knowledge-graph picture, answers with evidence, a "plain RAG vs GraphRAG" comparison, and a zip of all outputs.

*Tip: for a faster run, set `USE_LLM = False` in the cell called "Run the demo".*

## Step 1 — Install libraries

In [ ]:
!pip install -q networkx spacy scikit-learn matplotlib transformers sentencepiece
!python -m spacy download en_core_web_sm -q

## Step 2 — Create the project files (folders + code + case files)
The next cells just write the project's files into Colab. You don't need to read them.

In [ ]:
import os
for d in ['data/case_files', 'graphrag', 'outputs']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile data/entities.json
[
  {"name": "Helen Voss",  "type": "PERSON", "aliases": ["Voss", "Curator Voss"]},
  {"name": "Tomas Reed",  "type": "PERSON", "aliases": ["Reed"]},
  {"name": "Ravi Menon",  "type": "PERSON", "aliases": ["Menon", "Inspector Menon"]},
  {"name": "Nils Arden",  "type": "PERSON", "aliases": ["Arden"]},
  {"name": "Priya Nair",  "type": "PERSON", "aliases": ["Nair"]},
  {"name": "Felix Ortega","type": "PERSON", "aliases": ["Ortega"]},
  {"name": "Silas Crane", "type": "PERSON", "aliases": ["Crane"]},
  {"name": "Oona Park",   "type": "PERSON", "aliases": ["Park"]},
  {"name": "Greta Holm",  "type": "PERSON", "aliases": ["Holm"]},
  {"name": "Pavel Dunn",  "type": "PERSON", "aliases": ["Dunn"]},
  {"name": "Ines Calder", "type": "PERSON", "aliases": ["Calder"]},
  {"name": "Marlow Museum",     "type": "ORG", "aliases": ["the museum"]},
  {"name": "Marlow Police",     "type": "ORG", "aliases": []},
  {"name": "Harbor Catering",   "type": "ORG", "aliases": []},
  {"name": "Crane Auctions",    "type": "ORG", "aliases": []},
  {"name": "Sentinel Security", "type": "ORG", "aliases": []},
  {"name": "Marlow Library",    "type": "ORG", "aliases": ["the library"]},
  {"name": "Harbor Festival",   "type": "EVENT", "aliases": ["the festival"]},
  {"name": "Dockside Warehouse","type": "PLACE", "aliases": []},
  {"name": "Blue Heron Pearl",  "type": "ITEM", "aliases": []}
]

In [ ]:
%%writefile data/case_files/01_theft_report.txt
On the night of 12 March, the Blue Heron Pearl vanished from the Marlow Museum. Helen Voss discovered the empty display case at 7 a.m. and called Marlow Police. Inspector Ravi Menon took charge of the investigation. Night guard Tomas Reed told Menon that he saw a white van parked behind the museum at midnight. Reed said the alarm stayed silent all night. Helen Voss confirmed that the Blue Heron Pearl was insured for two million dollars.

In [ ]:
%%writefile data/case_files/02_the_van.txt
The white van belonged to Harbor Catering. Harbor Catering is owned by Nils Arden. Arden told Inspector Ravi Menon that he never lent the van to anyone. Harbor Catering had supplied the food for the museum gala on 10 March. Driver Priya Nair said she left the van keys with Felix Ortega after the gala. Nair added that Ortega promised to return the keys the next morning.

In [ ]:
%%writefile data/case_files/03_felix_ortega.txt
Felix Ortega worked as a technician at the Marlow Museum until January. Helen Voss fired Ortega after a dispute about missing equipment. After the dismissal, Ortega owed a large debt to Silas Crane. Silas Crane owns Crane Auctions. Crane Auctions hosted a private sale of pearl replicas in February. Ortega told friends that he would soon repay Crane.

In [ ]:
%%writefile data/case_files/04_alarm_log.txt
Sentinel Security installed the alarm system at the Marlow Museum. Technician Oona Park inspected the alarm on 13 March. Park wrote that the alarm was disabled with a maintenance code and was not forced. Park noted that only Tomas Reed and Felix Ortega ever received that code. Sentinel Security had given Ortega the code while he worked at the museum.

In [ ]:
%%writefile data/case_files/05_dockside_warehouse.txt
Silas Crane contacted Nils Arden about renting vans during the winter. Crane Auctions and Harbor Catering both rent space at Dockside Warehouse. Inspector Ravi Menon searched Dockside Warehouse on 15 March. Menon found the Blue Heron Pearl hidden inside a catering crate. The crate carried the label of Harbor Catering. Marlow Police arrested Felix Ortega at Dockside Warehouse. Crane denied any involvement and hired a lawyer.

In [ ]:
%%writefile data/case_files/06_library_circle.txt
Greta Holm runs the Reading Circle at the Marlow Library. Pavel Dunn helps Holm organise the yearly book fair. Helen Voss donated rare books to the Marlow Library for the book fair. Ines Calder opened the book fair with a short speech. Marlow Library hosts the fair every April.

In [ ]:
%%writefile data/case_files/07_harbor_festival.txt
Ines Calder organises the Harbor Festival every summer. Harbor Catering sponsors the Harbor Festival with free snacks. Nils Arden leads the festival food stalls. Pavel Dunn volunteers at the Harbor Festival as a stage manager. Greta Holm reads stories at the festival.

In [ ]:
%%writefile graphrag/__init__.py
from .build_graph import build_graph
from .communities import detect_communities, summarize_communities
from .query import GraphRAG
from .baseline import NaiveRAG

In [ ]:
%%writefile graphrag/build_graph.py
"""Step 1 of GraphRAG: turn raw text files into a knowledge graph.

Nodes  = entities (people, organisations, places, items)
Edges  = "these two entities appear in the same sentence"
         (edge keeps the sentences as evidence + the main verb as a label)
"""
import glob
import json
import os
import re
from itertools import combinations

import networkx as nx
import spacy

PRONOUN_STARTS = ("he ", "she ", "his ", "her ", "they ", "their ")


class EntityFinder:
    """Finds known entities (and their aliases) inside a piece of text."""

    def __init__(self, entities):
        self.type_of = {e["name"]: e["type"] for e in entities}
        self.alias_to_name = {}
        for e in entities:
            for alias in [e["name"], *e.get("aliases", [])]:
                self.alias_to_name[alias.lower()] = e["name"]
        # longest aliases first so "Crane Auctions" wins over "Crane"
        aliases = sorted(self.alias_to_name, key=len, reverse=True)
        self.pattern = re.compile(
            r"\b(" + "|".join(re.escape(a) for a in aliases) + r")\b", re.IGNORECASE
        )

    def find(self, text):
        found, seen = [], set()
        for m in self.pattern.finditer(text):
            name = self.alias_to_name[m.group(1).lower()]
            if name not in seen:
                seen.add(name)
                found.append(name)
        return found


def load_entities(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def read_documents(folder):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*.txt"))):
        with open(path, encoding="utf-8") as f:
            docs.append((os.path.basename(path), f.read()))
    return docs


def build_graph(data_dir="data", nlp=None):
    """Returns (graph, sentences, finder).

    sentences = list of dicts: {id, doc, text, entities}
    """
    entities = load_entities(os.path.join(data_dir, "entities.json"))
    finder = EntityFinder(entities)
    nlp = nlp or spacy.load("en_core_web_sm", disable=["ner"])

    G = nx.Graph()
    for e in entities:
        G.add_node(e["name"], type=e["type"], mentions=0)
    sentences = []

    for fname, text in read_documents(os.path.join(data_dir, "case_files")):
        prev_people = []
        for sent in nlp(text).sents:
            s = sent.text.strip()
            if not s:
                continue
            ents = finder.find(s)
            people = [e for e in ents if finder.type_of[e] == "PERSON"]

            # light "he/she" resolution: only when the previous sentence had exactly one person
            if not people and s.lower().startswith(PRONOUN_STARTS) and len(prev_people) == 1:
                ents.insert(0, prev_people[0])
                people = [prev_people[0]]
            prev_people = people

            root = next((t.lemma_.lower() for t in sent if t.dep_ == "ROOT"), "relates to")
            sid = len(sentences)
            sentences.append({"id": sid, "doc": fname, "text": s, "entities": ents, "verb": root})

            for e in ents:
                G.nodes[e]["mentions"] += 1
            for a, b in combinations(sorted(set(ents)), 2):
                if not G.has_edge(a, b):
                    G.add_edge(a, b, weight=0, relations=[], evidence=[])
                G[a][b]["weight"] += 1
                G[a][b]["relations"].append(root)
                G[a][b]["evidence"].append(sid)

    G.remove_nodes_from([n for n in list(G.nodes) if G.nodes[n]["mentions"] == 0])
    return G, sentences, finder

In [ ]:
%%writefile graphrag/communities.py
"""Step 2 of GraphRAG: find communities (groups) in the graph and summarise each one.

This is the 'Graph' idea from Microsoft's GraphRAG: instead of only searching single
sentences, we also keep a summary of every cluster so we can answer big-picture questions.
Summaries here are extractive (no LLM needed), so this step is fast and free.
"""
import networkx as nx


def detect_communities(G, seed=42):
    comms = nx.algorithms.community.louvain_communities(G, weight="weight", seed=seed)
    comms = sorted(comms, key=lambda c: (-len(c), sorted(c)[0]))
    for i, c in enumerate(comms):
        for n in c:
            G.nodes[n]["community"] = i
    return comms


def summarize_communities(G, comms, sentences, max_sentences=3):
    summaries = []
    for i, c in enumerate(comms):
        members = sorted(c, key=lambda n: (-G.degree(n, weight="weight"), n))
        scored = []
        for s in sentences:
            inside = [e for e in s["entities"] if e in c]
            # keep sentences that talk mostly about this group
            if len(inside) >= 2 or (len(inside) == 1 and len(s["entities"]) == 1):
                scored.append((len(inside), -s["id"], s))
        scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
        key_sentences = [s["text"] for _, _, s in scored[:max_sentences]]
        summaries.append(
            {
                "id": i,
                "title": " / ".join(members[:2]),
                "members": members,
                "key_sentences": key_sentences,
            }
        )
    return summaries

In [ ]:
%%writefile graphrag/baseline.py
"""A plain 'naive RAG' baseline: find the sentences most similar to the question (TF-IDF).
Used to show where GraphRAG does better (multi-hop questions)."""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class NaiveRAG:
    def __init__(self, sentences):
        self.sentences = sentences
        self.vec = TfidfVectorizer(stop_words="english")
        self.matrix = self.vec.fit_transform([s["text"] for s in sentences])

    def search(self, question, k=3):
        q = self.vec.transform([question])
        scores = cosine_similarity(q, self.matrix)[0]
        order = scores.argsort()[::-1][:k]
        return [self.sentences[i] for i in order if scores[i] > 0]

In [ ]:
%%writefile graphrag/llm.py
"""Optional free local language model (Google FLAN-T5) that writes the final answer.
It runs on the Colab CPU/GPU - no API key, no payment."""


def load_llm(model_name="google/flan-t5-base", max_new_tokens=140):
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()

    def generate(prompt):
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=3)
        return tok.decode(out[0], skip_special_tokens=True).strip()

    return generate

In [ ]:
%%writefile graphrag/viz.py
"""Draw the knowledge graph (coloured by community) and optionally highlight a path."""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx

PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2", "#9D755D"]


def draw_graph(G, path=None, out_file="outputs/graph.png", title="Case knowledge graph"):
    pos = nx.spring_layout(G, seed=7, k=0.9, weight="weight")
    plt.figure(figsize=(13, 9))
    colors = [PALETTE[G.nodes[n].get("community", 0) % len(PALETTE)] for n in G]
    sizes = [500 + 250 * G.degree(n) for n in G]
    widths = [0.8 + 0.9 * G[u][v]["weight"] for u, v in G.edges]
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.35, edge_color="#555555")
    if path and len(path) > 1:
        path_edges = list(zip(path[:-1], path[1:]))
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, width=4.5, edge_color="#D62728")
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, edgecolors="white")
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
    plt.title(title, fontsize=15)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_file, dpi=140)
    plt.close()
    return out_file

In [ ]:
%%writefile graphrag/query.py
"""Step 3 of GraphRAG: answer questions by walking the graph.

Three retrieval modes (chosen automatically):
  link     - question mentions 2+ entities  -> find the connection path between them (multi-hop)
  profile  - question mentions 1 entity     -> collect that entity's neighbourhood
  global   - big-picture question           -> use the community summaries
  keyword  - nothing recognised             -> fall back to TF-IDF sentence search
"""
import re

import networkx as nx

from .baseline import NaiveRAG

GLOBAL_WORDS = re.compile(
    r"\b(overall|main groups?|themes?|clusters?|big picture|summar\w+|communities|groups)\b", re.I
)


class GraphRAG:
    def __init__(self, G, sentences, summaries, finder, llm=None):
        self.G, self.sentences, self.summaries, self.finder, self.llm = (
            G, sentences, summaries, finder, llm,
        )
        self.fallback = NaiveRAG(sentences)

    # ---------- retrieval ----------
    def _edge_evidence(self, a, b, limit=2):
        ids = self.G[a][b]["evidence"][:limit]
        return [self.sentences[i] for i in ids]

    def retrieve(self, question):
        ents = [e for e in self.finder.find(question) if e in self.G]
        is_global = bool(GLOBAL_WORDS.search(question))

        if len(ents) >= 2:
            return self._link_mode(ents)
        if len(ents) == 1:
            return self._profile_mode(ents[0])
        if is_global:
            return self._global_mode()
        hits = self.fallback.search(question, k=4)
        return {"mode": "keyword", "entities": [], "path": [], "evidence": hits,
                "facts": [h["text"] for h in hits]}

    def _link_mode(self, ents):
        full_path, evidence = [], []
        for a, b in zip(ents[:-1], ents[1:]):
            if nx.has_path(self.G, a, b):
                p = nx.shortest_path(self.G, a, b)
            else:
                p = [a, b]
            full_path += p if not full_path else p[1:]
            for u, v in zip(p[:-1], p[1:]):
                if self.G.has_edge(u, v):
                    evidence += self._edge_evidence(u, v)
        seen, uniq = set(), []
        for s in evidence:
            if s["id"] not in seen:
                seen.add(s["id"])
                uniq.append(s)
        chain = " -> ".join(full_path)
        hops = max(len(full_path) - 1, 0)
        facts = [f"Connection chain ({hops} hop{'s' if hops != 1 else ''}): {chain}."]
        facts += [s["text"] for s in uniq]
        return {"mode": "link", "entities": ents, "path": full_path, "evidence": uniq, "facts": facts}

    def _profile_mode(self, ent):
        G = self.G
        neighbours = sorted(G[ent], key=lambda n: -G[ent][n]["weight"])
        mine = [s for s in self.sentences if ent in s["entities"]]
        mine.sort(key=lambda s: -len(s["entities"]))
        mine = mine[:6]
        group = self.summaries[G.nodes[ent]["community"]]
        facts = [f"{ent} is a {G.nodes[ent]['type'].lower()} directly linked to: {', '.join(neighbours)}."]
        facts += [s["text"] for s in mine]
        return {"mode": "profile", "entities": [ent], "path": [], "evidence": mine,
                "facts": facts, "group": group}

    def _global_mode(self):
        facts = []
        for s in self.summaries:
            if len(s["members"]) < 2:
                continue
            facts.append(f"Group {s['id'] + 1} ({', '.join(s['members'])}): " + " ".join(s["key_sentences"][:2]))
        return {"mode": "global", "entities": [], "path": [], "evidence": [], "facts": facts}

    # ---------- answering ----------
    def _template_answer(self, r):
        if r["mode"] == "link":
            return r["facts"][0] + " Evidence is listed below."
        if r["mode"] == "profile":
            return r["facts"][0]
        if r["mode"] == "global":
            return "The case files split into these groups:\n" + "\n".join("  - " + f for f in r["facts"])
        return "Closest matching sentences found (no entity recognised in the question)."

    def answer(self, question):
        r = self.retrieve(question)
        r["question"] = question
        r["template_answer"] = self._template_answer(r)
        if self.llm:
            context = "\n".join("- " + f for f in r["facts"])
            prompt = (
                "Answer the question using only the facts below. Be specific and name the "
                "people and organisations involved.\n\nFacts:\n" + context +
                f"\n\nQuestion: {question}\nAnswer:"
            )
            r["llm_answer"] = self.llm(prompt)
        return r


def format_result(r):
    lines = [f"Q: {r['question']}", f"Mode: {r['mode']}"]
    if r["path"]:
        lines.append("Path: " + " -> ".join(r["path"]))
    lines.append("Answer: " + r.get("llm_answer", r["template_answer"]))
    if r.get("llm_answer"):
        lines.append("Graph facts: " + r["template_answer"])
    if r["evidence"]:
        lines.append("Evidence:")
        lines += [f"  [{e['doc']}] {e['text']}" for e in r["evidence"]]
    return "\n".join(lines)

In [ ]:
%%writefile run_demo.py
"""CaseBoard GraphRAG - run everything:  python run_demo.py   (add --no-llm for a fast run)"""
import argparse
import os

from graphrag import GraphRAG, NaiveRAG, build_graph, detect_communities, summarize_communities
from graphrag.query import format_result
from graphrag.viz import draw_graph

DEMO_QUESTIONS = [
    "How is Felix Ortega connected to the Blue Heron Pearl?",
    "How is Silas Crane connected to the Marlow Museum?",
    "Tell me about Nils Arden.",
    "What are the main groups in these case files?",
    "Who could have disabled the alarm with a maintenance code?",
]
# <<< Add your own questions below (use names from data/entities.json) >>>
MY_QUESTIONS = []


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--no-llm", action="store_true", help="skip the language model (faster)")
    ap.add_argument("--model", default="google/flan-t5-base", help="HuggingFace model name")
    ap.add_argument("--question", help="ask one custom question")
    args = ap.parse_args()
    os.makedirs("outputs", exist_ok=True)

    print("1/4 Building knowledge graph ...")
    G, sentences, finder = build_graph("data")
    comms = detect_communities(G)
    summaries = summarize_communities(G, comms, sentences)
    print(f"    {G.number_of_nodes()} entities, {G.number_of_edges()} relationships, {len(comms)} communities")

    print("2/4 Saving graph picture + community report ...")
    draw_graph(G, out_file="outputs/graph.png")
    with open("outputs/communities.md", "w", encoding="utf-8") as f:
        f.write("# Communities found in the case files\n\n")
        for s in summaries:
            f.write(f"## Group {s['id'] + 1}: {s['title']}\n**Members:** {', '.join(s['members'])}\n\n")
            for k in s["key_sentences"]:
                f.write(f"- {k}\n")
            f.write("\n")

    llm = None
    if not args.no_llm:
        print("3/4 Loading free local language model (first time downloads ~1GB) ...")
        from graphrag.llm import load_llm
        llm = load_llm(args.model)
    else:
        print("3/4 Skipping language model (--no-llm)")

    print("4/4 Answering questions ...\n")
    engine = GraphRAG(G, sentences, summaries, finder, llm)
    questions = [args.question] if args.question else DEMO_QUESTIONS + MY_QUESTIONS
    report = ["# Answers\n"]
    for i, q in enumerate(questions):
        r = engine.answer(q)
        text = format_result(r)
        print(text + "\n" + "-" * 70)
        report.append("```\n" + text + "\n```\n")
        if r["path"]:
            draw_graph(G, path=r["path"], out_file=f"outputs/path_q{i + 1}.png",
                       title=f"Reasoning path: {q}")
    with open("outputs/answers.md", "w", encoding="utf-8") as f:
        f.write("\n".join(report))

    # GraphRAG vs plain RAG on the multi-hop question
    q = "How is Silas Crane connected to the Marlow Museum?"
    naive = NaiveRAG(sentences).search(q, k=3)
    graph = engine.retrieve(q)
    with open("outputs/compare_naive_vs_graphrag.md", "w", encoding="utf-8") as f:
        f.write(f"# Plain RAG vs GraphRAG\n\nQuestion: {q}\n\n## Plain RAG (top-3 similar sentences)\n")
        f.writelines(f"- [{s['doc']}] {s['text']}\n" for s in naive)
        f.write("\n## GraphRAG (follows the connections)\n")
        f.writelines(f"- {x}\n" for x in graph["facts"])
    print("Done! Open the 'outputs' folder: graph.png, path_q*.png, answers.md, communities.md, compare_naive_vs_graphrag.md")


if __name__ == "__main__":
    main()

## Step 3 — Run the demo

In [ ]:
USE_LLM = True   # True = also use the free FLAN-T5 model to phrase answers (first run downloads ~1 GB). False = faster.
flag = '' if USE_LLM else '--no-llm'
!python run_demo.py {flag}

## Step 4 — See the results

In [ ]:
from IPython.display import Image, display, Markdown
print('The knowledge graph (colours = communities):')
display(Image('outputs/graph.png'))
print('Reasoning path for: How is Silas Crane connected to the Marlow Museum?  (red = path)')
display(Image('outputs/path_q2.png'))
display(Markdown(open('outputs/compare_naive_vs_graphrag.md').read()))
display(Markdown(open('outputs/communities.md').read()))

## Step 5 — Ask your own question 🔎
Change the question and press ▶ on this cell. Use names from `data/entities.json` (e.g. Felix Ortega, Nils Arden, Helen Voss, Harbor Catering...).

In [ ]:
import importlib, sys
importlib.invalidate_caches()
sys.path.insert(0, '.')
from graphrag import build_graph, detect_communities, summarize_communities, GraphRAG
from graphrag.query import format_result
from graphrag.viz import draw_graph

MY_QUESTION = "How is Priya Nair connected to Silas Crane?"

G, sentences, finder = build_graph('data')
summaries = summarize_communities(G, detect_communities(G), sentences)
llm = None
if USE_LLM:
    from graphrag.llm import load_llm
    llm = load_llm()
engine = GraphRAG(G, sentences, summaries, finder, llm)
result = engine.answer(MY_QUESTION)
print(format_result(result))
if result['path']:
    draw_graph(G, path=result['path'], out_file='outputs/my_question.png', title=MY_QUESTION)
    display(Image('outputs/my_question.png'))

## Step 6 — Download everything

In [ ]:
import shutil
shutil.make_archive('caseboard_outputs', 'zip', 'outputs')
try:
    from google.colab import files
    files.download('caseboard_outputs.zip')
except Exception as e:
    print('Zip saved as caseboard_outputs.zip (use the Files panel on the left to download).')